# Phase 6.1: Environment Initialization and Data Flash-Extraction
Mounts Google Drive to get checkpoints and extracts the raw dataset zip file to local Colab storage.

In [ ]:
import os
import shutil
from google.colab import drive

# Mount Google Drive for persistent state tracking storage
drive.mount('/content/drive')


In [ ]:
!python --version

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
# # 1. Clean out the environment completely
# !pip uninstall -y mamba-ssm causal-conv1d tilelang quack-kernels
# !pip cache purge

# # 2. Make sure Ninja is ready for rapid multi-threading
# !pip install ninja

# # 3. Direct install Python 3.13 compatible versions with the "skip heavy dependencies" flags
# # (This will build using local PyTorch 2.11.0+cu128 in roughly 12-15 minutes total)
# %env MAMBA_SKIP_TILELANG=1
# %env MAMBA_SKIP_CUTLASS=1
# !pip install "causal-conv1d>=1.2.0" --no-build-isolation
# !pip install mamba-ssm==2.3.0 --no-build-isolation

# # 4. Create permanent Drive storage folder path
# !mkdir -p /content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/data/mamba_wheels

# # 5. Locate the freshly compiled Python 3.13 wheels from the cache and move them to Drive
# !find /root/.cache/pip/wheels/ -name "*.whl" -exec cp {} /content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/data/mamba_wheels/ \;

# # 6. Verify files are safe inside Drive folder
# !ls /content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/data/mamba_wheels


In [ ]:
# Install the pre-compiled files instantly from Drive
print("Installing mamba_wheels from drive...")
!pip install /content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/data/mamba_wheels/*.whl
print("Mamba_wheels installed.")

In [ ]:
!pip install -q monai

In [ ]:
# Define raw archive path and fast local runtime target destination paths
# DATASET_ZIP = "/content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/data/BraTS2020_TrainingData.zip"
# DATASET_ZIP = "/content/drive/MyDrive/ML-Datasets/BraTS2020_TrainingData_128.zip"
# DATASET_ZIP = "/content/drive/MyDrive/ML-Datasets/BraTS2020_TrainingData_128_Cubic_B_spline.zip"
DATASET_ZIP = "/content/drive/MyDrive/ML-Datasets/BraTS2020_TrainingData_ABC.zip"
LOCAL_EXTRACT_DIR = "/content/MICCAI_BraTS2020_TrainingData_ABC"
# LOCAL_DATA_DIR = os.path.join(LOCAL_EXTRACT_DIR, "MICCAI_BraTS2020_TrainingData_128")
LOCAL_DATA_DIR = LOCAL_EXTRACT_DIR

# Verify archive existence immediately before runtime allocation
assert os.path.exists(DATASET_ZIP), f"Dataset archive not found: {DATASET_ZIP}"

# Robust extraction guard: triggers if directory does not exist or is completely empty
# if not os.path.exists(LOCAL_EXTRACT_DIR) or len(os.listdir(LOCAL_EXTRACT_DIR)) == 0:
if not os.path.exists(LOCAL_DATA_DIR) or len(os.listdir(LOCAL_DATA_DIR)) == 0:
    print(f"Extracting preprocessed dataset to fast local runtime storage: {LOCAL_EXTRACT_DIR}...")
    os.makedirs(LOCAL_EXTRACT_DIR, exist_ok=True)
    shutil.unpack_archive(DATASET_ZIP, LOCAL_EXTRACT_DIR, "zip")
    print("Extraction complete. Preprocessed dataset ready for I/O operations.")
else:
    print("Valid local preprocessed dataset cache detected. Skipping extraction.")

# Component Integration and Framework Imports

In [ ]:
import sys
import random
import numpy as np
import torch
import torch.optim as optim
import itertools

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Optimize CUDA runtime convolution algorithm selection for static tensor patches
torch.backends.cudnn.benchmark = True

# Add src to path
PROJECT_ROOT = "/content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor"
sys.path.append(PROJECT_ROOT)

import src.config as config
from src.dataset import get_brats_dataloaders
from src.losses import SegmentationLoss
from src.engine import run_training

# Instantiate the 5 distinct sub-network modules forming the end-to-end multi-task architecture
from src.models.mamba_backbone import MambaBackbone
from src.models.fusion import PresenceAwareCrossModalFusion
from src.models.mamba_backbone import SharedDeepMambaBackbone
from src.models.decoder import SegmentationDecoder3D, AuxiliaryDecoder3D

In [ ]:
import importlib

importlib.reload(config)

import src.engine as engine
importlib.reload(engine)

import src.models.mamba_backbone as mbb
importlib.reload(mbb)

import src.models.fusion as mf
importlib.reload(mf)

import src.models.decoder as md
importlib.reload(md)

import src.losses as sl
importlib.reload(sl)

import src.transforms as tf
importlib.reload(tf)

# Execution

In [ ]:
from torch.amp import GradScaler

# Hardware runtime verification
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Operational Hardware target identified: {device}")
print(f"Target Checkpoint Saving Directory: {config.CHECKPOINT_DIR}")

# 1. Pipeline Dataset Loaders Construction
print("Instantiating MONAI dictionary data pipelines...")
train_loader, val_loader = get_brats_dataloaders(batch_size = 8, is_joint_training=True) # defauld set to batch_size=config.BATCH_SIZE
# ----------------------------------------------

# 2. Structural Module Instantiations
print("Initializing neural net components...")
backbone = MambaBackbone(embed_dim=config.EMBED_DIM).to(device)
fusion = PresenceAwareCrossModalFusion(embed_dim=config.EMBED_DIM).to(device)
shared_backbone = SharedDeepMambaBackbone(embed_dim=config.EMBED_DIM).to(device)
decoder = SegmentationDecoder3D(embed_dim=config.EMBED_DIM, out_channels=config.NUM_SEG_CLASSES).to(device)
aux_decoder = AuxiliaryDecoder3D(embed_dim=config.EMBED_DIM, out_channels=config.NUM_SEG_CLASSES).to(device)
# ----------------------------------------------

# Load Pretrained Expert Weights
latest_ckpt_path = os.path.join(config.CHECKPOINT_DIR, "latest_checkpoint.pth")

if os.path.exists(latest_ckpt_path):
    print("[*] Existing joint-training checkpoint detected. Bypassing expert loading to resume from latest state.")
else:
    existing_experts = [path for path in config.PRETRAINED_EXPERT_PATHS.values() if os.path.exists(path)]

    if len(existing_experts) == len(config.PRETRAINED_EXPERT_PATHS):
        print("[*] All pre-trained expert weights found. Initiating loading process...")
        backbone.load_expert_weights(config.PRETRAINED_EXPERT_PATHS, device=device)
    elif len(existing_experts) > 0:
        print(f"[*] Found {len(existing_experts)}/4 pre-trained expert weights. Loading available weights...")
        backbone.load_expert_weights(config.PRETRAINED_EXPERT_PATHS, device=device)
    else:
        print("[!] No pre-trained expert weights found. Proceeding with standard random initialization.")
# ----------------------------------------------

model_components = (backbone, fusion, shared_backbone, decoder, aux_decoder)

# Compute total parameter profile summary metrics for publication tracking
total_params = sum(p.numel() for model in model_components for p in model.parameters())
print(f"Total Multi-Task Trainable Network Parameters: {total_params:,}")

# 3. Unified Multi-Task Optimization System Setup
criterion = SegmentationLoss().to(device)

# # Chain layer parameters
# all_parameters = itertools.chain(
#     backbone.parameters(),
#     fusion.parameters(),
#     shared_backbone.parameters(),
#     decoder.parameters(),
#     aux_decoder.parameters()
# )

# optimizer = optim.AdamW(
#     all_parameters,
#     lr=config.LEARNING_RATE,
#     weight_decay=config.WEIGHT_DECAY
# )

# Define separate parameter groups for Differential Learning Rates
optimizer = optim.AdamW(
    [
        {'params': backbone.parameters(), 'lr': config.ENCODER_LR},
        {'params': fusion.parameters(), 'lr': config.BASE_LR},
        {'params': shared_backbone.parameters(), 'lr': config.BASE_LR},
        {'params': decoder.parameters(), 'lr': config.BASE_LR},
        {'params': aux_decoder.parameters(), 'lr': config.BASE_LR}
    ],
    weight_decay=config.WEIGHT_DECAY)

scheduler = optim.lr_scheduler.PolynomialLR(
    optimizer,
    total_iters=config.TOTAL_EPOCHS,
    power=0.9
)

# Device-aware mixed-precision gradient scaling framework
scaler = GradScaler(enabled=(device.type == "cuda"))
# ----------------------------------------------

# 4. Trigger Orchestration Engine Pipeline
print("Handing execution loop management over to stateful runtime engine...")
run_training(
    model_components=model_components,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    scaler=scaler,
    device=device,
    session_epoch=41
)
# ----------------------------------------------